In [219]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import pulp
from pulp import LpProblem

I=24
J=15
K=35

n=1000000
mois_mapping = {
    'Janvier': 1,
    'Février': 2,
    'Mars': 3,
    'Avril': 4,
    'Mai': 5,
    'Juin': 6,
    'Juillet': 7,
    'Aout': 8,
    'Septembre': 9,
    'Octobre': 10,
    'Novembre': 11,
    'Décembre': 12
}

prices = pd.read_csv('Prices.csv').values
production=pd.read_csv('Production.csv')
production['Mois'] = production['Mois'].map(mois_mapping)
production = np.delete(production.values,2,axis=1)
surface = pd.read_csv('Simulation.csv').values
charges_var= np.delete(pd.read_csv('Charges_var.csv').values,[1,2,3,4,7,8],axis=1)

def get_week(j):
    month=production[j][4]
    year = 2024
    first_day = datetime(year, month, 1)
    if month == 12:
        last_day = datetime(year, 12, 31)
    else:
        last_day = datetime(year, month + 1, 1) - timedelta(days=1)
    weeks = []
    current_day = first_day
    while current_day <= last_day:
        week_num = current_day.isocalendar()[1]
        if week_num == 1 and current_day.month == 12:
            week_num = 53
        if week_num not in weeks:
            week_start = current_day - timedelta(days=current_day.weekday())
            week_end = week_start + timedelta(days=6)
            days_in_current_month = sum(1 for i in range(7) if (week_start + timedelta(days=i)).month == month)
            if days_in_current_month > 3:
                weeks.append(week_num)
        current_day += timedelta(days=7)
    new_weeks=[]
    for week in weeks:
        if week != 48:
            new_weeks.append(week - 14)
    return new_weeks
print(get_week(4))

[0, 1, 2, 3]


In [220]:
C_A = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
C_V = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
B = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
for i in range(I):
    for j in range(J):
        for k in get_week(j):
            var=0
            for t in range(37):
                p=np.where(prices==production[j][1])[0][0]
                if (t+k+production[j][5]+15<91):
                    # les index ajoutés dans les listes ne sont que pour le parcours des fichiers csv
                    var+=production[j][t+7]*surface[i][2]*prices[p][t+15+k+(production[j][5])] # 15 survient du fait que l'indexation de k commence à partir du 1er Avril 2024 et l'ajout de 1 pour le parcours de la liste
                else:
                    #a=int((t+14+k+production[j][5])%90)
                    #var+=production[j][t+7]*surface[i][2]*prices[p][a]
                    var+=0 #Veuillez commenter cette ligne et décommenter la précedente pour obtenir l'hyptohese de redondance des prix
            C_A[i][j][k]=var
            C_V[i][j][k]=(surface[i][2]*charges_var[j][2])
            B[i][j][k]=C_A[i][j][k]-C_V[i][j][k]

In [221]:
prob = pulp.LpProblem("Max_Profit", pulp.LpMaximize)
X = pulp.LpVariable.dicts("X", (range(I), range(J), range(K)), cat=pulp.LpBinary)
prob += pulp.lpSum(B[i][j][k] * X[i][j][k] for i in range(I) for j in range(J) for k in get_week(j))
for i in range(I):
    prob += pulp.lpSum(X[i][j][k] for j in range(J) for k in get_week(j)) == 1
    for k in range(K):
        if (i!=21) and (i!=22) and (i!=23):
            if(k in get_week(4)):
                prob += X[i][4][k]==0
            if(k in get_week(3)):
                prob += X[i][3][k]==0
        for j in range(J):
            if(j==9 or j==10 or j==11 or j==13):
                if(k in get_week(j)):
                    if (i!=19) and (i!=20):
                        prob += X[i][j][k]==0
                    if(surface[i][2]>2.87):
                        prob += X[i][j][k]==0 
# Pour le bonus veuillez décommenter la ligne ci-dessus                       
#prob += pulp.lpSum(production[j][t+7] * X[i][j][k] for t in range(37) for i in range(I) for j in range(J) for k in get_week(j))<=n 
prob

Max_Profit:
MAXIMIZE
107150.0*X_0_0_10 + 119800.0*X_0_0_11 + 142550.0*X_0_0_12 + 96300.0*X_0_0_9 + -211500.0*X_0_10_31 + -187100.0*X_0_10_32 + -177300.0*X_0_10_33 + -100000000.0*X_0_11_26 + -100000000.0*X_0_11_27 + -100000000.0*X_0_11_28 + -100000000.0*X_0_11_29 + -100000000.0*X_0_11_30 + 357750.0*X_0_12_4 + 296275.0*X_0_12_5 + 232900.0*X_0_12_6 + 180500.0*X_0_12_7 + 140025.0*X_0_12_8 + 167200.0*X_0_13_26 + 106650.0*X_0_13_27 + 56250.0*X_0_13_28 + 26500.0*X_0_13_29 + 32450.0*X_0_13_30 + 241750.0*X_0_14_26 + 177150.0*X_0_14_27 + 123200.0*X_0_14_28 + 91475.0*X_0_14_29 + 97900.0*X_0_14_30 + 95325.0*X_0_1_13 + 120250.0*X_0_1_14 + 139975.0*X_0_1_15 + 154825.0*X_0_1_16 + 21250.0*X_0_2_17 + 36000.0*X_0_2_18 + 50850.0*X_0_2_19 + 73600.0*X_0_2_20 + 85900.0*X_0_2_21 + 194925.0*X_0_3_4 + 142500.0*X_0_3_5 + 98875.0*X_0_3_6 + 65400.0*X_0_3_7 + 56450.0*X_0_3_8 + 359625.0*X_0_4_0 + 329725.0*X_0_4_1 + 289475.0*X_0_4_2 + 246200.0*X_0_4_3 + -81925.0*X_0_5_10 + -73375.0*X_0_5_11 + -67825.0*X_0_5_12 + -94

In [222]:
prob.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/rachidaitjalloul/anaconda3/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/zb/_d8dsvp17n54wwjbqt8lknz80000gn/T/a01be7f59f89492fbcbebab7de862fca-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/zb/_d8dsvp17n54wwjbqt8lknz80000gn/T/a01be7f59f89492fbcbebab7de862fca-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 614 COLUMNS
At line 7536 RHS
At line 8146 BOUNDS
At line 9731 ENDATA
Problem MODEL has 609 rows, 1584 columns and 2169 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.12066e+07 - 0.00 seconds
Cgl0002I 585 variables fixed
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from -1.12066e+07 to -1.79

1

In [217]:
print("Statut de la solution:", prob.status)
print("Valeur optimale de la fonction objectif:", prob.objective.value())


Statut de la solution: 1
Valeur optimale de la fonction objectif: 11206575.0


In [218]:
for v in prob.variables():
    if (v.varValue==1):
        print(v.name, "=", v.varValue)
        serre=int(v.name.split("_")[1])+1
        scenario=production[int(v.name.split("_")[2])][0]
        semaine=int(v.name.split("_")[3])+14
        print("Le résultats est dans la serre {}, scenario {}, et semaine {} ".format(serre, scenario,semaine))

X_0_12_4 = 1.0
Le résultats est dans la serre 1, scenario 15, et semaine 18 
X_10_12_4 = 1.0
Le résultats est dans la serre 11, scenario 15, et semaine 18 
X_11_12_4 = 1.0
Le résultats est dans la serre 12, scenario 15, et semaine 18 
X_12_12_4 = 1.0
Le résultats est dans la serre 13, scenario 15, et semaine 18 
X_13_12_4 = 1.0
Le résultats est dans la serre 14, scenario 15, et semaine 18 
X_14_12_4 = 1.0
Le résultats est dans la serre 15, scenario 15, et semaine 18 
X_15_12_4 = 1.0
Le résultats est dans la serre 16, scenario 15, et semaine 18 
X_16_12_4 = 1.0
Le résultats est dans la serre 17, scenario 15, et semaine 18 
X_17_12_4 = 1.0
Le résultats est dans la serre 18, scenario 15, et semaine 18 
X_18_12_4 = 1.0
Le résultats est dans la serre 19, scenario 15, et semaine 18 
X_19_12_4 = 1.0
Le résultats est dans la serre 20, scenario 15, et semaine 18 
X_1_12_4 = 1.0
Le résultats est dans la serre 2, scenario 15, et semaine 18 
X_20_12_4 = 1.0
Le résultats est dans la serre 21, scena